In [ ]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. Environment and Database Connection
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# 2. Fetch data from the CLEANED table
df = pd.read_sql("SELECT * FROM cleaned_housing_data", engine)

def feature_engineering(df):
    """
    Transforms data using insights from EDA (01): 
    New features, Binary flags, and Log transformations.
    """
    df = df.copy()

    # --- A. DERIVED FEATURES (Sizin əvvəlki sütunlarınız) ---
    df['house_age'] = df['yr_sold'] - df['year_built']
    df['total_living_area'] = df['first_flr_sf'] + df['second_flr_sf'] + df['total_bsmt_sf']
    df['is_remodeled'] = (df['year_built'] != df['year_remod_add']).astype(int)
    df['total_bathrooms'] = (
        df['full_bath'] + df['half_bath'] * 0.5 +
        df['bsmt_full_bath'] + df['bsmt_half_bath'] * 0.5
    )

    # --- B. BINARY FLAGS (01-də yaratdığınız "Var/Yox" məntiqi) ---
    # Sıfırı çox olan sütunlar üçün modelə "bu xüsusiyyət varmı?" siqnalını veririk.
    zero_cols = ['mas_vnr_area', 'wood_deck_sf', 'open_porch_sf', 'bsmt_fin_sf_1']
    for col in zero_cols:
        if col in df.columns:
            df[f'has_{col}'] = (df[col] > 0).astype(int)

    # --- C. LOG TRANSFORMATIONS (01-dəki ən kritik hissə) ---
    # Skewness-i (asimmetriyanı) azaltmaq üçün log1p tətbiq edirik.
    # log1p = log(1 + x) -> bu, sıfır olan dəyərlərdə xəta almamaq üçündür.
    log_cols = [
        'gr_liv_area', 'lot_frontage', 'total_living_area', 
        'garage_area', 'bsmt_unf_sf', 'lot_area'
    ] + zero_cols
    
    for col in log_cols:
        if col in df.columns:
            df[col] = np.log1p(df[col])

    # --- D. CLEANUP ---
    # Artıq ehtiyac olmayan və ya təkrarçılıq yaradan sütunları silirik
    cols_to_drop = [
        'year_remod_add', 
        'first_flr_sf', 'second_flr_sf', 'total_bsmt_sf',
        'full_bath', 'half_bath', 'bsmt_full_bath', 'bsmt_half_bath'
    ]
    df = df.drop(columns=cols_to_drop, errors='ignore')

    return df

# Prosesi icra edirik
df_engineered = feature_engineering(df)

# 3. Save the result to SQL
df_engineered.to_sql('engineered_housing_data', engine, if_exists='replace', index=False)

print("Step 03: Feature engineering complete. Log transformations and binary flags applied.")